# Final Threshold Selection and Unbiased Test Evaluation

## Overview & Purpose
This notebook addresses a critical methodological requirement by performing **out-of-sample threshold optimization**. To prevent subtle test-set data leakage, the operational classification decision boundary is selected strictly on the **calibration dataset**, frozen, and then evaluated once on the untouched **held-out test dataset**.

### Key Objectives:
1. **Artifact Ingestion & Audit:** Load saved model artifacts and dataset splits, verifying strict 3-way partition integrity (\(\text{Train}\), \(\text{Calibration}\), \(\text{Test}\)).
2. **Calibration-Set Threshold Optimization:** Evaluate classification performance across decision boundaries ($0.05$ to $0.95$ in steps of $0.05$) exclusively using calibration-set probabilities to identify the optimal threshold (e.g., maximum \(F_1\) score).
3. **Threshold Freezing:** Lock the selected optimal threshold without looking at or optimizing against test set predictions.
4. **Unbiased Test Set Evaluation:** Apply the frozen threshold to the held-out test set to compute unbiased classification metrics:
   - **Accuracy**
   - **Balanced Accuracy**
   - **Precision**
   - **Recall (Sensitivity)**
   - **Specificity**
   - **\(F_1\) Score**
   - **Confusion Matrix**
5. **Baseline Comparison:** Contrast final test-set performance at the newly selected threshold directly against the standard default threshold ($0.50$).
6. **Artifact Export:** Save all corrected threshold evaluation tables and summary metrics to feed into the final project report in Notebook 11.

In [2]:
import os
import joblib
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    log_loss,
    confusion_matrix
)

print("NOTEBOOK 12 - FINAL THRESHOLD SELECTION")


NOTEBOOK 12 - FINAL THRESHOLD SELECTION


In [4]:
import os
import joblib

# Dynamically locate the correct artifact folder
possible_paths = [
    os.path.join("notebook8_outputs", "final_artifacts"),
    os.path.join("outputs", "final_artifacts"),
    os.path.join("final_artifacts")
]

ARTIFACT_DIR = None
for path in possible_paths:
    if os.path.exists(path):
        ARTIFACT_DIR = path
        break

if ARTIFACT_DIR is None:
    ARTIFACT_DIR = os.path.join("notebook8_outputs", "final_artifacts")

OUTPUT_DIR_12 = os.path.join("outputs", "notebook_12")
os.makedirs(OUTPUT_DIR_12, exist_ok=True)

print("Artifact directory resolved to:", os.path.abspath(ARTIFACT_DIR))
print("Notebook 12 output directory:", os.path.abspath(OUTPUT_DIR_12))

# Verify file availability
required_files = [
    "final_base_model.joblib",
    "X_model_train.joblib",
    "y_model_train.joblib",
    "X_calibration.joblib",
    "y_calibration.joblib",
    "X_test.joblib",
    "y_test.joblib",
    "train_prob.joblib",
    "calibration_prob.joblib",
    "test_prob.joblib"
]

missing_files = [
    f for f in required_files
    if not os.path.exists(os.path.join(ARTIFACT_DIR, f))
]

if missing_files:
    raise FileNotFoundError(
        f"Missing required artifacts in {ARTIFACT_DIR}: {missing_files}"
    )

# Load artifacts
final_base_model = joblib.load(os.path.join(ARTIFACT_DIR, "final_base_model.joblib"))
X_model_train = joblib.load(os.path.join(ARTIFACT_DIR, "X_model_train.joblib"))
y_model_train = joblib.load(os.path.join(ARTIFACT_DIR, "y_model_train.joblib"))

X_calibration = joblib.load(os.path.join(ARTIFACT_DIR, "X_calibration.joblib"))
y_calibration = joblib.load(os.path.join(ARTIFACT_DIR, "y_calibration.joblib"))

X_test = joblib.load(os.path.join(ARTIFACT_DIR, "X_test.joblib"))
y_test = joblib.load(os.path.join(ARTIFACT_DIR, "y_test.joblib"))

train_prob = joblib.load(os.path.join(ARTIFACT_DIR, "train_prob.joblib"))
calibration_prob = joblib.load(os.path.join(ARTIFACT_DIR, "calibration_prob.joblib"))
test_prob = joblib.load(os.path.join(ARTIFACT_DIR, "test_prob.joblib"))

print("✓ All artifacts loaded successfully.")

Artifact directory resolved to: c:\Users\rayan\OneDrive\Desktop\Credit_Risk\Credit-Risk-Scoring-System\notebooks\notebook8_outputs\final_artifacts
Notebook 12 output directory: c:\Users\rayan\OneDrive\Desktop\Credit_Risk\Credit-Risk-Scoring-System\notebooks\outputs\notebook_12
✓ All artifacts loaded successfully.


#### Verify dataset and probability alignment

In [5]:
print("=" * 75)
print("DATA AND PROBABILITY VERIFICATION")
print("=" * 75)

assert X_model_train.shape[0] == len(y_model_train) == len(train_prob)
assert X_calibration.shape[0] == len(y_calibration) == len(calibration_prob)
assert X_test.shape[0] == len(y_test) == len(test_prob)

assert X_model_train.shape[1] == 23
assert X_calibration.shape[1] == 23
assert X_test.shape[1] == 23

assert np.isfinite(train_prob).all()
assert np.isfinite(calibration_prob).all()
assert np.isfinite(test_prob).all()

assert ((train_prob >= 0) & (train_prob <= 1)).all()
assert ((calibration_prob >= 0) & (calibration_prob <= 1)).all()
assert ((test_prob >= 0) & (test_prob <= 1)).all()

print(f"Model training: {X_model_train.shape}")
print(f"Calibration:     {X_calibration.shape}")
print(f"Test:            {X_test.shape}")

print("\nProbability alignment verified.")
print("Probability range verified.")
print("All verification checks passed.")

DATA AND PROBABILITY VERIFICATION
Model training: (19200, 23)
Calibration:     (4800, 23)
Test:            (6000, 23)

Probability alignment verified.
Probability range verified.
All verification checks passed.


#### Confirm calibration and test separation

In [6]:
print("=" * 75)
print("CALIBRATION / TEST SEPARATION")
print("=" * 75)

assert len(calibration_prob) == 4800
assert len(test_prob) == 6000

print(f"Calibration observations: {len(calibration_prob):,}")
print(f"Test observations:       {len(test_prob):,}")

print(
    f"Calibration default rate: "
    f"{np.mean(y_calibration):.6f}"
)

print(
    f"Test default rate: "
    f"{np.mean(y_test):.6f}"
)

print("\nCalibration data will be used for threshold selection.")
print("Test data will remain untouched until final evaluation.")

CALIBRATION / TEST SEPARATION
Calibration observations: 4,800
Test observations:       6,000
Calibration default rate: 0.221250
Test default rate: 0.221167

Calibration data will be used for threshold selection.
Test data will remain untouched until final evaluation.


#### Define threshold evaluation function

In [7]:
def evaluate_threshold(y_true, probabilities, threshold):
    predictions = (probabilities >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        predictions,
        labels=[0, 1]
    ).ravel()

    specificity = tn / (tn + fp)

    return {
        "Threshold": threshold,
        "Accuracy": accuracy_score(y_true, predictions),
        "Balanced Accuracy": balanced_accuracy_score(y_true, predictions),
        "Precision": precision_score(
            y_true,
            predictions,
            zero_division=0
        ),
        "Recall": recall_score(
            y_true,
            predictions,
            zero_division=0
        ),
        "Specificity": specificity,
        "F1": f1_score(
            y_true,
            predictions,
            zero_division=0
        ),
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp
    }

#### Analyze thresholds on calibration set

In [8]:
print("=" * 75)
print("CALIBRATION-SET THRESHOLD ANALYSIS")
print("=" * 75)

thresholds = np.round(
    np.arange(0.05, 0.951, 0.01),
    2
)

calibration_threshold_results = []

for threshold in thresholds:
    result = evaluate_threshold(
        y_calibration,
        calibration_prob,
        threshold
    )
    calibration_threshold_results.append(result)

calibration_threshold_df = pd.DataFrame(
    calibration_threshold_results
)

print(
    f"Thresholds evaluated: "
    f"{len(calibration_threshold_df)}"
)

print(
    calibration_threshold_df[
        [
            "Threshold",
            "Balanced Accuracy",
            "Precision",
            "Recall",
            "F1",
            "Specificity"
        ]
    ].head()
)

CALIBRATION-SET THRESHOLD ANALYSIS
Thresholds evaluated: 91
   Threshold  Balanced Accuracy  Precision    Recall        F1  Specificity
0       0.05           0.528840   0.231865  0.981168  0.375090     0.076512
1       0.06           0.550156   0.240530  0.974576  0.385834     0.125736
2       0.07           0.567567   0.248535  0.958569  0.394727     0.176565
3       0.08           0.589215   0.259107  0.951036  0.407258     0.227394
4       0.09           0.607477   0.269915  0.928437  0.418240     0.286517


#### Select threshold using calibration set

In [9]:
best_calibration_row = (
    calibration_threshold_df
    .sort_values(
        by=["F1", "Balanced Accuracy", "Threshold"],
        ascending=[False, False, True]
    )
    .iloc[0]
)

SELECTED_THRESHOLD = float(
    best_calibration_row["Threshold"]
)

print("=" * 75)
print("SELECTED THRESHOLD")
print("=" * 75)

print(
    f"Selected threshold from calibration set: "
    f"{SELECTED_THRESHOLD:.2f}"
)

print(
    f"Calibration F1: "
    f"{best_calibration_row['F1']:.6f}"
)

print(
    f"Calibration Balanced Accuracy: "
    f"{best_calibration_row['Balanced Accuracy']:.6f}"
)

print(
    f"Calibration Precision: "
    f"{best_calibration_row['Precision']:.6f}"
)

print(
    f"Calibration Recall: "
    f"{best_calibration_row['Recall']:.6f}"
)

SELECTED THRESHOLD
Selected threshold from calibration set: 0.31
Calibration F1: 0.539526
Calibration Balanced Accuracy: 0.701417
Calibration Precision: 0.567568
Calibration Recall: 0.514124


#### Show calibration threshold table around the selected threshold

In [10]:
window = calibration_threshold_df[
    calibration_threshold_df["Threshold"].between(
        max(0.05, SELECTED_THRESHOLD - 0.05),
        min(0.95, SELECTED_THRESHOLD + 0.05)
    )
].copy()

display(
    window[
        [
            "Threshold",
            "Balanced Accuracy",
            "Precision",
            "Recall",
            "Specificity",
            "F1"
        ]
    ].reset_index(drop=True)
)

,Threshold,Balanced Accuracy,Precision,Recall,Specificity,F1
0,0.26,0.704235,0.506329,0.564972,0.843499,0.534045
1,0.27,0.703872,0.517575,0.554614,0.853130,0.535455
2,0.28,0.704247,0.531193,0.545198,0.863296,0.538104
3,0.29,0.703220,0.539262,0.536723,0.869716,0.537990
4,0.30,0.700845,0.548228,0.524482,0.877207,0.536092
5,0.31,0.701417,0.567568,0.514124,0.888711,0.539526
6,0.32,0.699518,0.575725,0.504708,0.894329,0.537883
7,0.33,0.693928,0.575691,0.490584,0.897271,0.529741
8,0.34,0.692703,0.584282,0.483051,0.902354,0.528866
9,0.35,0.688390,0.585280,0.471751,0.905029,0.522419


#### Freeze threshold and evaluate untouched test set

In [11]:
print("=" * 75)
print("FINAL TEST-SET EVALUATION")
print("=" * 75)

test_predictions = (
    test_prob >= SELECTED_THRESHOLD
).astype(int)

tn, fp, fn, tp = confusion_matrix(
    y_test,
    test_predictions,
    labels=[0, 1]
).ravel()

test_specificity = tn / (tn + fp)

final_test_metrics = {
    "Threshold": SELECTED_THRESHOLD,
    "Accuracy": accuracy_score(
        y_test,
        test_predictions
    ),
    "Balanced Accuracy": balanced_accuracy_score(
        y_test,
        test_predictions
    ),
    "Precision": precision_score(
        y_test,
        test_predictions,
        zero_division=0
    ),
    "Recall": recall_score(
        y_test,
        test_predictions,
        zero_division=0
    ),
    "Specificity": test_specificity,
    "F1": f1_score(
        y_test,
        test_predictions,
        zero_division=0
    ),
    "ROC-AUC": roc_auc_score(
        y_test,
        test_prob
    ),
    "PR-AUC": average_precision_score(
        y_test,
        test_prob
    ),
    "Brier Score": brier_score_loss(
        y_test,
        test_prob
    ),
    "Log Loss": log_loss(
        y_test,
        test_prob
    ),
    "TN": tn,
    "FP": fp,
    "FN": fn,
    "TP": tp
}

final_test_metrics_df = pd.DataFrame(
    [final_test_metrics]
)

display(final_test_metrics_df)

FINAL TEST-SET EVALUATION


,Threshold,Accuracy,Balanced Accuracy,Precision,Recall,Specificity,F1,ROC-AUC,PR-AUC,Brier Score,Log Loss,TN,FP,FN,TP
0,0.31,0.800667,0.700173,0.552442,0.51997,0.880377,0.535714,0.780532,0.560152,0.135042,0.429563,4114,559,637,690


#### Final confusion matrix

In [12]:
print("=" * 75)
print("FINAL TEST CONFUSION MATRIX")
print("=" * 75)

final_confusion_matrix = pd.DataFrame(
    [[tn, fp], [fn, tp]],
    index=[
        "Actual Non-Default",
        "Actual Default"
    ],
    columns=[
        "Predicted Non-Default",
        "Predicted Default"
    ]
)

display(final_confusion_matrix)

FINAL TEST CONFUSION MATRIX


,Predicted Non-Default,Predicted Default
Actual Non-Default,4114,559
Actual Default,637,690


#### Compare corrected threshold with 0.50

In [13]:
baseline_threshold = 0.50

baseline_result = evaluate_threshold(
    y_test,
    test_prob,
    baseline_threshold
)

selected_result = evaluate_threshold(
    y_test,
    test_prob,
    SELECTED_THRESHOLD
)

threshold_comparison = pd.DataFrame([
    {
        "Threshold": baseline_threshold,
        "Accuracy": baseline_result["Accuracy"],
        "Balanced Accuracy": baseline_result["Balanced Accuracy"],
        "Precision": baseline_result["Precision"],
        "Recall": baseline_result["Recall"],
        "Specificity": baseline_result["Specificity"],
        "F1": baseline_result["F1"],
        "TN": baseline_result["TN"],
        "FP": baseline_result["FP"],
        "FN": baseline_result["FN"],
        "TP": baseline_result["TP"]
    },
    {
        "Threshold": SELECTED_THRESHOLD,
        "Accuracy": selected_result["Accuracy"],
        "Balanced Accuracy": selected_result["Balanced Accuracy"],
        "Precision": selected_result["Precision"],
        "Recall": selected_result["Recall"],
        "Specificity": selected_result["Specificity"],
        "F1": selected_result["F1"],
        "TN": selected_result["TN"],
        "FP": selected_result["FP"],
        "FN": selected_result["FN"],
        "TP": selected_result["TP"]
    }
])

display(threshold_comparison)

,Threshold,Accuracy,Balanced Accuracy,Precision,Recall,Specificity,F1,TN,FP,FN,TP
0,0.50,0.819333,0.655770,0.668985,0.362472,0.949069,0.470186,4435,238,846,481
1,0.31,0.800667,0.700173,0.552442,0.519970,0.880377,0.535714,4114,559,637,690


#### Calculate final predicted default rate

In [14]:
print("=" * 75)
print("FINAL CLASSIFICATION COUNTS")
print("=" * 75)

predicted_default_count = int(
    np.sum(test_predictions)
)

predicted_non_default_count = int(
    len(test_predictions) - predicted_default_count
)

print(
    f"Predicted non-default: "
    f"{predicted_non_default_count:,}"
)

print(
    f"Predicted default: "
    f"{predicted_default_count:,}"
)

print(
    f"Predicted default proportion: "
    f"{predicted_default_count / len(test_predictions):.6f}"
)

print(
    f"Actual default proportion: "
    f"{np.mean(y_test):.6f}"
)

FINAL CLASSIFICATION COUNTS
Predicted non-default: 4,751
Predicted default: 1,249
Predicted default proportion: 0.208167
Actual default proportion: 0.221167


#### Save corrected threshold results

In [17]:
calibration_threshold_path = os.path.join(
    OUTPUT_DIR_12,
    "calibration_threshold_analysis.csv"
)

final_metrics_path = os.path.join(
    OUTPUT_DIR_12,
    "final_unbiased_test_metrics.csv"
)

confusion_matrix_path = os.path.join(
    OUTPUT_DIR_12,
    "final_unbiased_confusion_matrix.csv"
)

threshold_comparison_path = os.path.join(
    OUTPUT_DIR_12,
    "final_threshold_comparison.csv"
)

calibration_threshold_df.to_csv(
    calibration_threshold_path,
    index=False
)

final_test_metrics_df.to_csv(
    final_metrics_path,
    index=False
)

final_confusion_matrix.to_csv(
    confusion_matrix_path
)

threshold_comparison.to_csv(
    threshold_comparison_path,
    index=False
)

print("=" * 75)
print("FILES SAVED")
print("=" * 75)

print(calibration_threshold_path)
print(final_metrics_path)
print(confusion_matrix_path)
print(threshold_comparison_path)

FILES SAVED
outputs\notebook_12\calibration_threshold_analysis.csv
outputs\notebook_12\final_unbiased_test_metrics.csv
outputs\notebook_12\final_unbiased_confusion_matrix.csv
outputs\notebook_12\final_threshold_comparison.csv


#### Final methodological verification

In [16]:
print("=" * 75)
print("FINAL METHODOLOGICAL VERIFICATION")
print("=" * 75)

assert SELECTED_THRESHOLD >= 0.05
assert SELECTED_THRESHOLD <= 0.95

assert len(calibration_threshold_df) == 91

assert len(test_predictions) == 6000

assert (
    final_test_metrics_df.loc[0, "Threshold"]
    == SELECTED_THRESHOLD
)

assert (
    final_confusion_matrix.values.sum()
    == 6000
)

assert np.isfinite(
    final_test_metrics_df[
        [
            "Accuracy",
            "Balanced Accuracy",
            "Precision",
            "Recall",
            "Specificity",
            "F1",
            "ROC-AUC",
            "PR-AUC",
            "Brier Score",
            "Log Loss"
        ]
    ].values
).all()

print("✓ Threshold selected using calibration set")
print("✓ Threshold frozen before test evaluation")
print("✓ Test set contains 6,000 observations")
print("✓ Test confusion matrix totals 6,000")
print("✓ Final metrics are finite")
print("✓ Final methodological verification passed")

FINAL METHODOLOGICAL VERIFICATION
✓ Threshold selected using calibration set
✓ Threshold frozen before test evaluation
✓ Test set contains 6,000 observations
✓ Test confusion matrix totals 6,000
✓ Final metrics are finite
✓ Final methodological verification passed


#### Summary

In [15]:
print("=" * 75)
print("NOTEBOOK 12 - FINAL SUMMARY")
print("=" * 75)

print()
print("Threshold Selection")
print("-" * 75)
print(
    f"Selected using calibration set: "
    f"{SELECTED_THRESHOLD:.2f}"
)

print()
print("Untouched Test-Set Performance")
print("-" * 75)

print(
    f"Accuracy:           "
    f"{final_test_metrics_df.loc[0, 'Accuracy']:.6f}"
)

print(
    f"Balanced Accuracy:  "
    f"{final_test_metrics_df.loc[0, 'Balanced Accuracy']:.6f}"
)

print(
    f"Precision:          "
    f"{final_test_metrics_df.loc[0, 'Precision']:.6f}"
)

print(
    f"Recall:             "
    f"{final_test_metrics_df.loc[0, 'Recall']:.6f}"
)

print(
    f"Specificity:        "
    f"{final_test_metrics_df.loc[0, 'Specificity']:.6f}"
)

print(
    f"F1:                 "
    f"{final_test_metrics_df.loc[0, 'F1']:.6f}"
)

print(
    f"ROC-AUC:            "
    f"{final_test_metrics_df.loc[0, 'ROC-AUC']:.6f}"
)

print(
    f"PR-AUC:             "
    f"{final_test_metrics_df.loc[0, 'PR-AUC']:.6f}"
)

print(
    f"Brier Score:        "
    f"{final_test_metrics_df.loc[0, 'Brier Score']:.6f}"
)

print(
    f"Log Loss:            "
    f"{final_test_metrics_df.loc[0, 'Log Loss']:.6f}"
)

print()
print("Final test confusion matrix")
print("-" * 75)
display(final_confusion_matrix)

print()
print("Notebook 12 completed successfully.")

NOTEBOOK 12 - FINAL SUMMARY

Threshold Selection
---------------------------------------------------------------------------
Selected using calibration set: 0.31

Untouched Test-Set Performance
---------------------------------------------------------------------------
Accuracy:           0.800667
Balanced Accuracy:  0.700173
Precision:          0.552442
Recall:             0.519970
Specificity:        0.880377
F1:                 0.535714
ROC-AUC:            0.780532
PR-AUC:             0.560152
Brier Score:        0.135042
Log Loss:            0.429563

Final test confusion matrix
---------------------------------------------------------------------------


,Predicted Non-Default,Predicted Default
Actual Non-Default,4114,559
Actual Default,637,690



Notebook 12 completed successfully.
